In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

In [3]:
sentences = [
    "tigers are running in forest",
    "birds are flying in sky", 
    "river flows in forest", 
    "mountains are still", 
    "humans are walking in park",
    "people are eating food"
]

In [4]:
labels = [0, 0, 1, 1, 2, 2]
# Labels:
# 0 = animals in motion
# 1 = static nature
# 2 = human activity

In [5]:
vocab = set()

In [6]:
type(vocab)

set

In [7]:
vocab

set()

In [8]:
for i in sentences:
    print(i)

tigers are running in forest
birds are flying in sky
river flows in forest
mountains are still
humans are walking in park
people are eating food


In [9]:
for i in sentences:
    for everysent in i.split():
        print(everysent)
    

tigers
are
running
in
forest
birds
are
flying
in
sky
river
flows
in
forest
mountains
are
still
humans
are
walking
in
park
people
are
eating
food


In [10]:
for sentence in sentences:
    for word in sentence.split():
        vocab.add(word)

In [11]:
vocab

{'are',
 'birds',
 'eating',
 'flows',
 'flying',
 'food',
 'forest',
 'humans',
 'in',
 'mountains',
 'park',
 'people',
 'river',
 'running',
 'sky',
 'still',
 'tigers',
 'walking'}

In [12]:
vocab = list(vocab)

In [13]:
vocab

['still',
 'tigers',
 'food',
 'flows',
 'birds',
 'mountains',
 'people',
 'park',
 'flying',
 'walking',
 'river',
 'are',
 'forest',
 'in',
 'humans',
 'eating',
 'sky',
 'running']

In [14]:
word_to_idx = {word: i for i, word in enumerate(vocab)}

In [15]:
word_to_idx

{'still': 0,
 'tigers': 1,
 'food': 2,
 'flows': 3,
 'birds': 4,
 'mountains': 5,
 'people': 6,
 'park': 7,
 'flying': 8,
 'walking': 9,
 'river': 10,
 'are': 11,
 'forest': 12,
 'in': 13,
 'humans': 14,
 'eating': 15,
 'sky': 16,
 'running': 17}

In [16]:
test = torch.zeros(len(vocab))

In [17]:
test

tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

In [18]:
def sentence_to_vector(sentence):
    vector = torch.zeros(len(vocab))
    for word in sentence.split():
        if word in word_to_idx:
            vector[word_to_idx[word]] = 1
    return vector

In [19]:
X = torch.stack([sentence_to_vector(s) for s in sentences])

In [20]:
X

tensor([[0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 1., 1., 0., 0., 0., 1.],
        [0., 0., 0., 0., 1., 0., 0., 0., 1., 0., 0., 1., 0., 1., 0., 0., 1., 0.],
        [0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 1., 0., 1., 1., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 1., 0., 1., 0., 1., 0., 1., 1., 0., 0., 0.],
        [0., 0., 1., 0., 0., 0., 1., 0., 0., 0., 0., 1., 0., 0., 0., 1., 0., 0.]])

In [21]:
y = torch.tensor(labels)

In [22]:
y

tensor([0, 0, 1, 1, 2, 2])

In [23]:
class SimpleNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(SimpleNN, self).__init__()
        
        # Input → Hidden layer
        self.fc1 = nn.Linear(input_size, hidden_size)
        
        # Activation function
        self.relu = nn.ReLU()
        
        # Hidden → Output layer
        self.fc2 = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        # Pass through first layer
        out = self.fc1(x)
        
        # Apply activation
        out = self.relu(out)
        
        # Output layer
        out = self.fc2(out)
        
        return out

In [24]:
# Initialize model
myowngpt = SimpleNN(input_size=len(vocab), hidden_size=8, output_size=3)

In [25]:
# ================================
# 6. LOSS + OPTIMIZER
# ================================
criterion = nn.CrossEntropyLoss()   # For classification
optimizer = optim.Adam(myowngpt.parameters(), lr=0.01)

In [26]:
# ================================
# 7. TRAINING LOOP
# ================================
print("=== TRAINING START ===")

for epoch in range(100):
    
    # Forward pass
    outputs = myowngpt(X)
    
    # Compute loss
    loss = criterion(outputs, y)
    
    # Backward pass (compute gradients)
    optimizer.zero_grad()
    loss.backward()
    
    # Update weights
    optimizer.step()
    
    if (epoch+1) % 20 == 0:
        print(f"Epoch [{epoch+1}/100], Loss: {loss.item():.4f}")

print("=== TRAINING COMPLETE ===\n")


=== TRAINING START ===
Epoch [20/100], Loss: 0.6375
Epoch [40/100], Loss: 0.1405
Epoch [60/100], Loss: 0.0204
Epoch [80/100], Loss: 0.0068
Epoch [100/100], Loss: 0.0041
=== TRAINING COMPLETE ===



In [27]:
# ================================
# 8. TESTING / INFERENCE
# ================================

def predict(sentence):
    myowngpt.eval()  # Set model to evaluation mode
    
    vector = sentence_to_vector(sentence)
    
    with torch.no_grad():
        output = myowngpt(vector)
        predicted_class = torch.argmax(output).item()
    
    return predicted_class


In [28]:
test_sentence = "lions are running in jungle"
prediction = predict(test_sentence)

In [29]:
label_map = {
    0: "Animals in motion",
    1: "Static nature",
    2: "Human activity"
}

In [30]:
print("Test Sentence:", test_sentence)
print("Predicted Class:", prediction)
print("Meaning:", label_map[prediction])

Test Sentence: lions are running in jungle
Predicted Class: 0
Meaning: Animals in motion


In [31]:
test_sentence = "machines moving in city"
prediction = predict(test_sentence)

In [32]:
print("Test Sentence:", test_sentence)
print("Predicted Class:", prediction)
print("Meaning:", label_map[prediction])

Test Sentence: machines moving in city
Predicted Class: 1
Meaning: Static nature
